# 04 · FinBERT sentiment per filing section

**Reads** `stg_clean_10Ks` (dbt) · **Writes** `fact_finbert_sentiment` (one row per filing × section)

Fixes compared with the previous version of this notebook:

1. Class order is read from `model.config.id2label` (**positive, negative, neutral** for `ProsusAI/finbert`); the
   old code assumed positive, neutral, negative and therefore swapped negative and neutral.
2. Long sections are split into 510-token chunks and their probabilities averaged (evenly sampled, at most
   `max_chunks` per section) instead of scoring only the first 512 tokens.
3. The output is built and written in one place. The old notebook wrote a `sentiment_df` it never created.

Use the probabilities as *signals*, not labels: regulatory text is cautious by design.

In [ ]:
%pip install transformers torch

In [ ]:
%restart_python

In [ ]:
# Make the repository's Python package importable (Databricks Git folder: cwd is this notebook's folder)
import os, sys
ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from trading_sentiment import table

In [ ]:
dbutils.widgets.text("max_chunks", "32", "Max 510-token chunks scored per section")
MAX_CHUNKS = int(dbutils.widgets.get("max_chunks"))

import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

from trading_sentiment.sentiment import chunk_token_ids, finbert_summary, mean_probabilities

MODEL_NAME = "ProsusAI/finbert"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.model_max_length = 10**9            # we chunk ourselves; silence the "longer than 512" warning
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME).eval()
ID2LABEL = model.config.id2label
print(ID2LABEL)

In [ ]:
def score_text(text, batch_size=16):
    """Average FinBERT class probabilities over chunks covering the whole text."""
    ids = tokenizer(text, add_special_tokens=False, truncation=False)["input_ids"]
    chunks = chunk_token_ids(ids, max_chunks=MAX_CHUNKS)
    probs = []
    for i in range(0, len(chunks), batch_size):
        seqs = [[tokenizer.cls_token_id] + c + [tokenizer.sep_token_id] for c in chunks[i:i + batch_size]]
        width = max(len(q) for q in seqs)
        batch = {
            "input_ids": torch.tensor([q + [tokenizer.pad_token_id] * (width - len(q)) for q in seqs]),
            "attention_mask": torch.tensor([[1] * len(q) + [0] * (width - len(q)) for q in seqs]),
        }
        with torch.no_grad():
            logits = model(**batch).logits
        probs.extend(torch.softmax(logits, dim=-1).tolist())
    return {**finbert_summary(mean_probabilities(probs), ID2LABEL), "chunks_scored": len(chunks)}

In [ ]:
from pyspark.sql import functions as F

sections = (spark.table(table("stg_clean_10Ks"))
    .selectExpr("accessionNumber as accession_number", "cik", "company_name", "filing_date",
                "stack(3, 'BUSINESS', bus_text, 'RISKS', risks_text, 'MDA', mda_text) as (section_name, section_text)")
    .where("section_text is not null and length(trim(section_text)) > 0"))
pdf = sections.toPandas()

rows = []
for rec in pdf.itertuples():
    rows.append({"accession_number": rec.accession_number, "cik": rec.cik, "company_name": rec.company_name,
                 "filing_date": rec.filing_date, "section_name": rec.section_name, **score_text(rec.section_text)})
print(len(rows), "sections scored")

In [ ]:
schema = ("accession_number string, cik string, company_name string, filing_date date, section_name string, "
          "positive_prob double, negative_prob double, neutral_prob double, sentiment_label string, "
          "sentiment_score double, chunks_scored int")
fact = spark.createDataFrame(rows, schema).withColumn("processed_timestamp", F.current_timestamp())
fact.write.format("delta").mode("overwrite").saveAsTable(table("fact_finbert_sentiment"))
display(fact.groupBy("section_name", "sentiment_label").count().orderBy("section_name", "sentiment_label"))

The legacy `fact_sentiment` table (single score per filing, wrong class order) is no longer written; drop it once no
report reads it: `DROP TABLE IF EXISTS workspace.sec_filings.fact_sentiment`.